# Laboratory 3 — Hebbian Association and STDP

Companion to **Chapter 3: Hebbian Learning and Synaptic Plasticity**.

This controlled pairing example uses dimensionless activities and externally supplied output patterns. It illustrates partial-cue recall and ambiguity, **not autonomous associative learning**. The STDP window is a separate illustration; it does not train the association matrix.

**Workflow:** define model → validate an independent default baseline → edit experiment parameters → plot and interpret. Baseline checks intentionally do **not** depend on student-edited pairings or time constants.

[Run in Google Colab](https://colab.research.google.com/github/an-hongyu/Introduction-to-Neuromorphic-Embodied-AI/blob/main/notebooks/hebbian_learning.ipynb)

## 1. Model definition

For each explicitly paired input and output, use the weight update $\Delta W=\eta\,\mathbf y\mathbf x^\top$. For partial cues, compute output scores as $S=X_{\mathrm{cues}}W^\top$. The STDP function below is a **separate** time-window teaching rule, not the optimizer for the learned matrix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def train_association(patterns, paired_outputs, repetitions=8, learning_rate=0.1):
    """Apply an explicit Hebbian pairing list; W rows correspond to outputs."""
    X = np.asarray(patterns, dtype=float)
    Y = np.asarray(paired_outputs, dtype=float)
    if X.ndim != 2 or Y.ndim != 2 or X.shape[0] != Y.shape[0]:
        raise ValueError("Input and output pairings must have equal row counts.")
    if repetitions < 0 or int(repetitions) != repetitions or not np.isfinite(learning_rate) or learning_rate < 0:
        raise ValueError("Use nonnegative learning rate and integer repetitions.")
    W = np.zeros((Y.shape[1], X.shape[1]), dtype=float)
    for _ in range(int(repetitions)):
        for x, y in zip(X, Y):
            W += learning_rate * np.outer(y, x)
    return W

def pair_stdp(delta_ms, a_plus=0.01, a_minus=0.012, tau_ms=20.0):
    """Illustrative pair STDP, delta = t_post - t_pre (milliseconds)."""
    if not (np.isfinite(tau_ms) and tau_ms > 0):
        raise ValueError("STDP time constant must be positive.")
    delta_ms = np.asarray(delta_ms, dtype=float)
    return np.where(delta_ms > 0, a_plus * np.exp(-delta_ms/tau_ms),
                    np.where(delta_ms < 0, -a_minus * np.exp(delta_ms/tau_ms), 0.0))

## 2. Baseline validation — fixed reference only

The reference checks below use their **own** original input-output pairings and an explicit 20 ms STDP time constant. Do not use student-edited variables to validate fixed numerical examples: later edits should change the experimental output, **not** fail this reference check.

In [ ]:
BASELINE_PATTERNS = np.array([[1., 1., 0.], [0., 1., 1.]])
BASELINE_OUTPUTS = np.eye(2)
BASELINE_CUES = np.array([[1., 0., 0.], [0., 0., 1.], [0., 1., 0.]])
baseline_weights = train_association(BASELINE_PATTERNS, BASELINE_OUTPUTS)
baseline_scores = BASELINE_CUES @ baseline_weights.T
assert np.allclose(baseline_weights, [[0.8, 0.8, 0.], [0., 0.8, 0.8]])
assert np.allclose(baseline_scores, [[0.8, 0.], [0., 0.8], [0.8, 0.8]])
assert np.isclose(pair_stdp(10, tau_ms=20.), 0.01 * np.exp(-10/20))
assert pair_stdp(-10, tau_ms=20.) < 0 and pair_stdp(0, tau_ms=20.) == 0
print("Baseline checks passed for the original pairings and 20 ms STDP.")
print("Before learning, cue scores are zero; after learning:\n", baseline_scores)

## 3. Editable student experiment

Edit the next cell, then **Run All**. For Experiment 2, reverse the output rows with np.eye(2)[::-1]. For Experiment 4, try 10 and 40 ms for STDP_TAU_MS. The baseline check must still pass; the editable experiment should show different learned weights, recall scores, or STDP curves as appropriate.

To study interference, introduce an additional pairing in both the input and output matrices. The checks below verify parameter-aware mathematical properties, **not** one particular row ordering.

In [ ]:
# EDIT THESE to conduct your own experiments.
patterns = np.array([[1., 1., 0.], [0., 1., 1.]])
paired_outputs = np.eye(2)       # Try np.eye(2)[::-1] to swap output pairings.
cues = np.array([[1., 0., 0.], [0., 0., 1.], [0., 1., 0.]])
REPETITIONS = 8
LEARNING_RATE = 0.1
STDP_TAU_MS = 20.0               # Try 10.0 and 40.0.

weights = train_association(patterns, paired_outputs, REPETITIONS, LEARNING_RATE)
if cues.ndim != 2 or cues.shape[1] != weights.shape[1]:
    raise ValueError("Cue feature dimension must match the trained weights.")
scores = cues @ weights.T
assert weights.shape == (paired_outputs.shape[1], patterns.shape[1])
assert np.isfinite(weights).all() and np.isfinite(scores).all()
assert np.isclose(pair_stdp(10, tau_ms=STDP_TAU_MS),
                  0.01 * np.exp(-10.0 / STDP_TAU_MS))
assert pair_stdp(10, tau_ms=STDP_TAU_MS) > 0
assert pair_stdp(-10, tau_ms=STDP_TAU_MS) < 0
assert pair_stdp(0, tau_ms=STDP_TAU_MS) == 0

print("Weights (output rows × input features):\n", weights)
print("Partial-cue recall scores:\n", scores)
print(f"STDP tau = {STDP_TAU_MS:g} ms, delta-w(+10 ms) = "
      f"{float(pair_stdp(10, tau_ms=STDP_TAU_MS)):.6f}")
print("Shared features may create ambiguous scores. These are not probabilities.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
upper = max(0.8, float(np.max(weights)))
im = axes[0].imshow(weights, vmin=0, vmax=upper, cmap='Blues')
axes[0].set(xticks=np.arange(weights.shape[1]),
            xticklabels=['Feature 1', 'Shared', 'Feature 3'] if weights.shape[1] == 3 else
            [f'Feature {i+1}' for i in range(weights.shape[1])],
            yticks=np.arange(weights.shape[0]),
            yticklabels=[f'Output {i+1}' for i in range(weights.shape[0])],
            title='A. Learned weights')
for row in range(weights.shape[0]):
    for col in range(weights.shape[1]):
        axes[0].text(col, row, '%.1f' % weights[row, col], ha='center', va='center',
                     color='white' if weights[row, col] > upper/2 else '#17334c')
fig.colorbar(im, ax=axes[0], shrink=0.7, label='Weight (a.u.)')

x = np.arange(cues.shape[0])
width = 0.75 / scores.shape[1]
for j in range(scores.shape[1]):
    axes[1].bar(x - 0.375 + (j + 0.5)*width, scores[:, j], width,
                label=f'Output {j+1}')
axes[1].set(xticks=x, xticklabels=['Feature 1', 'Feature 3', 'Shared'] if len(cues) == 3
            else [f'Cue {i+1}' for i in x],
            ylabel='Output score (a.u.)', title='B. Partial-cue recall')
axes[1].legend(fontsize=8)

delta = np.linspace(-80, 80, 801)
for mask, color in [(delta < 0, '#b6483b'), (delta > 0, '#268471')]:
    axes[2].plot(delta[mask], pair_stdp(delta[mask], tau_ms=STDP_TAU_MS), color=color)
axes[2].scatter([0], [0], color='0.3', s=15, zorder=3)
axes[2].axhline(0, color='0.6', lw=0.7)
axes[2].axvline(0, color='0.6', lw=0.7)
axes[2].set(xlabel='Post minus pre spike time (ms)', ylabel='Weight change (a.u.)',
            title=f'C. STDP window (tau={STDP_TAU_MS:g} ms)')
plt.show()

## 4. Experiments and interpretation

1. **Compare before and after:** compare the zero-weight score before pairing with the learned scores for three partial cues.
2. **Shuffle output pairings:** set paired_outputs = np.eye(2)[::-1] and compare the two output rows and recall responses. Explain why the reference validation still passes.
3. **Test interference:** add an overlapping input pattern and its assigned output. Explain why a shared feature may activate more than one output.
4. **Change the STDP time constant:** compare 10, 20, and 40 ms. Inspect the timing window and the magnitude at both positive and negative 10 ms. Explain why the sign at a fixed nonzero time difference stays the same.

**The reference test answers whether the declared baseline was reproduced. The editable experiment tests a new choice of parameters.** Correctly changed weights or spike-timing windows should not be treated as code failures.

## References

[1] Donald O. Hebb, *The Organization of Behavior: A Neuropsychological Theory*. New York, NY, USA: Wiley, 1949. [Catalog record](https://wellcomecollection.org/works/g4sfzum4).

[2] Guo-qiang Bi and Mu-ming Poo, “Synaptic modifications in cultured hippocampal neurons: Dependence on spike timing, synaptic strength, and postsynaptic cell type,” *The Journal of Neuroscience*, vol. 18, no. 24, pp. 10464–10472, 1998, doi: [10.1523/JNEUROSCI.18-24-10464.1998](https://pubmed.ncbi.nlm.nih.gov/9852584/).